# Masowa imputacja: model GLM (Kim et al. 2021)

Estymator MI-K: $\hat{\mu}_{y,\text{MI-K}} = \sum_{i \in S_B} d_i^B \hat{y}_i / \sum_{i \in S_B} d_i^B$

**Procedura**:
1. Dopasuj model na próbie nielosowej $S_A$
2. Predykuj $\hat{y}$ na próbie losowej $S_B$
3. Oblicz ważoną średnią z wagami $d^B$

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
admin = pd.read_csv('../data/admin.csv')
jvs = pd.read_csv('../data/jvs.csv')

## region jako tekst zero-padded (zgodnie z R)
admin['region'] = admin['region'].astype(str).str.zfill(2)
jvs['region'] = jvs['region'].astype(str).str.zfill(2)

w_jvs = jvs['weight'].values

In [ ]:
## Funkcja pomocnicza: dopasowanie GLM i predykcja MI
def mi_glm(admin, jvs, formula_vars, cat_vars, target, family='binomial'):
    admin_dum = pd.get_dummies(admin[formula_vars], columns=cat_vars,
                               dtype=float, drop_first=True)
    jvs_dum = pd.get_dummies(jvs[formula_vars], columns=cat_vars,
                             dtype=float, drop_first=True)
    all_cols = sorted(set(admin_dum.columns) | set(jvs_dum.columns))
    admin_dum = admin_dum.reindex(columns=all_cols, fill_value=0)
    jvs_dum = jvs_dum.reindex(columns=all_cols, fill_value=0)

    X_admin = sm.add_constant(admin_dum).values.astype(float)
    X_jvs = sm.add_constant(jvs_dum).values.astype(float)
    y_admin = admin[target].values

    fam = sm.families.Binomial() if family == 'binomial' else sm.families.Gaussian()
    model = sm.GLM(y_admin, X_admin, family=fam).fit()
    return model, model.predict(X_jvs)

## Przykład 1: model regresji liniowej (gaussian) ~size

In [ ]:
model, y_hat = mi_glm(admin, jvs, ['size'], ['size'],
                      'single_shift', family='gaussian')
mu_glm1 = np.sum(w_jvs * y_hat) / np.sum(w_jvs)
print(f'MI-GLM (gauss, ~size):            {mu_glm1:.4f}')

## Przykład 2: model regresji logistycznej (binomial) ~size

Ponieważ `single_shift` jest zmienną binarną (0/1), model logistyczny jest bardziej naturalny.

In [ ]:
model, y_hat = mi_glm(admin, jvs, ['size'], ['size'],
                      'single_shift', family='binomial')
mu_glm2 = np.sum(w_jvs * y_hat) / np.sum(w_jvs)
print(f'MI-GLM (binom, ~size):            {mu_glm2:.4f}')

## Przykład 3: pełny model logistyczny ze wszystkimi zmiennymi

In [ ]:
model, y_hat = mi_glm(admin, jvs,
                      ['size', 'nace', 'region', 'private'],
                      ['size', 'nace', 'region'],
                      'single_shift', family='binomial')
mu_glm3 = np.sum(w_jvs * y_hat) / np.sum(w_jvs)
print(f'MI-GLM (binom, pełny):            {mu_glm3:.4f}')

print('\nWspółczynniki:')
print(model.params.round(4))

## Porównanie

In [ ]:
print(pd.DataFrame({
    'Metoda': ['MI-GLM gauss (~size)',
               'MI-GLM binom (~size)',
               'MI-GLM binom (pełny)'],
    'Oszacowanie': [round(mu_glm1, 4), round(mu_glm2, 4), round(mu_glm3, 4)]
}))

## Ćwiczenie

- Porównaj wyniki MI-GLM (binomial) z MI-NN i MI-PMM z notatnika `06-mi-nn.ipynb`
- Dodaj kolejne zmienne pomocnicze i sprawdź jak zmienia się oszacowanie
- Porównaj estymator MI-GLM z estymatorami IPW (notatnik `03-ipw-1.ipynb`)